In [32]:
import numpy as np
import wfdb
import os
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, lfilter
from scipy.fft import rfft, rfftfreq

records = "Person_01"

# Butterworth Filter Order 2 :
def butter2(sig) :
    l, h = 5, 15
    b, a = butter(
        N=2,
        Wn=[l,h],
        btype="bandpass",
        fs=500
    )
    filt = filtfilt(b, a, sig)
    return filt

def butter4(sig) :
    l, h = 5, 15
    b, a = butter(
        N=4,
        Wn=[l,h],
        btype="bandpass",
        fs=500
    )
    filt = filtfilt(b, a, sig)
    return filt    

def ma(sig) :
    M=10
    filt = np.convolve(sig, np.ones(M)/M, mode="same")
    return filt
    
def wma(sig) :
    wgts = np.array([1, 2, 3, 2, 1])
    wgts = wgts/np.sum(wgts)
    filt = np.convolve(sig, wgts, mode="same")
    return filt

def ema(sig) :
    alpha = 0.4
    filt = np.zeros(len(sig))
    filt[0] = sig[0]
    for n in range(1, len(sig)) :
        filt[n] = alpha*sig[n] + (1-alpha)*filt[n-1]
    return filt    

def bp(sig) :
    X = np.fft.fft(sig)
    N = len(sig)
    fs = 500
    freqs = np.fft.fftfreq(N, d=1/fs)
    l, h = 0.5, 50
    mask = (np.abs(freqs) >= l) & (np.abs(freqs) <= h)
    Xfil = X*mask
    filt = np.real(np.fft.ifft(Xfil))
    return filt

def gauss(sig) : 
    sigma = 2
    M = sigma*6
    x = np.linspace(-(M//2), M//2, M)
    wgts = np.exp(-x**2 / (2*sigma**2))
    wgts = wgts/np.sum(wgts)
    filt = np.convolve(sig, wgts, mode='same')
    return filt

filters = {
    "Butterworth4" : butter4,
    "Butterworth2" : butter2,
    "Moving Average" : ma,
    "Weighted Moving Average" : wma,
    "Exponential Moving Average" : ema,
    "Bandpass" : bp,
    "Gaussian" : gauss
}

#Energy Ratio
def enrat(sig, filtsig):
    raw_energy = np.sum(sig**2)
    filt_energy = np.sum(filtsig**2)
    return filt_energy/raw_energy
    
def snr_imp(raw, filtered):
    raw = np.asarray(raw)
    filtered = np.asarray(filtered)
    # Noise removed by filtering
    noise_removed = raw - filtered
    # Power of filtered signal and removed noise
    signal_power = np.sum(filtered**2)
    noise_power = np.sum(noise_removed**2) + 1e-12  # avoid divide by zero   
    # SNR improvement
    snr_imp = 10 * np.log10(signal_power / noise_power)
    return snr_imp
    
def ent_imp(raw, filt):
    def spec_entropy(sig):
        P = np.abs(rfft(sig))**2
        P = P / (np.sum(P) + 1e-12)
        return -np.sum(P * np.log2(P + 1e-12))
    return spec_entropy(raw) - spec_entropy(filt)
    
def ppr(raw, filt):
    raw_peaks = np.sum(raw > (np.mean(raw) + 0.5*np.std(raw)))
    filt_peaks = np.sum(filt > (np.mean(filt) + 0.5*np.std(filt)))
    return filt_peaks / (raw_peaks + 1e-12)
    
summary = {fi : [] for fi in filters}

sorted_records = [ f.replace(".hea", "")
                   for f in os.listdir(records)
                   if f.endswith(".hea") ]
sorted_records = sorted(sorted_records)


for rec in sorted_records :
    rec_path = os.path.join(records, rec)

    try:
        r = wfdb.rdrecord(rec_path)
        sig = r.p_signal[:, 0]
        for fn, ff in filters.items() :
            filt = ff(sig)

            snr = snr_imp(sig, filt)
            ent = ent_imp(sig, filt)
            peak = ppr(sig, filt)
            enr = enrat(sig, filt)

            summary[fn].append([enr, snr, ent, peak])
    except Exception as e:
        print("Skipping", rec, "->", e)
      

In [33]:
print("\n===== Average Results for Person_01 =====")

for f in summary:
    arr = np.array(summary[f])

    print(f"\n{f}")
    print("Average Energy:", np.mean(arr[:,0]))
    print("Average SNR:", np.mean(arr[:,1]))
    print("Average Entropy:", np.mean(arr[:,2]))
    print("Average PPR:", np.mean(arr[:,3]))


===== Average Results for Person_01 =====

Butterworth4
Average Energy: 0.18345811823479843
Average SNR: -6.513493805409669
Average Entropy: -0.7357861090792903
Average PPR: 0.7198339295649758

Butterworth2
Average Energy: 0.17027271498921734
Average SNR: -6.636535190617343
Average Entropy: -0.8674000445012158
Average PPR: 0.6916996361981109

Moving Average
Average Energy: 0.7331041746336054
Average SNR: 6.260099000127658
Average Entropy: 0.6270019116927603
Average PPR: 0.8940629470384435

Weighted Moving Average
Average Energy: 0.8960724881910741
Average SNR: 16.63577411664852
Average Entropy: 0.20915516608334847
Average PPR: 0.9443425142989416

Exponential Moving Average
Average Energy: 0.8486069689740049
Average SNR: 9.98368383500536
Average Entropy: 0.2651410717841828
Average PPR: 0.9291414082427725

Bandpass
Average Energy: 0.5756160148869331
Average SNR: 1.4093805157557537
Average Entropy: -1.6448576107058934
Average PPR: 0.790462720077087

Gaussian
Average Energy: 0.82326560142